# 02 — Feature Engineering

**Purpose:** Build the complete feature table for all 48 teams and all training / prediction rows.
Every feature group is constructed and verified here. No new logic is introduced — all computation
delegates to `src/features.py`.

**Outputs**
- `outputs/team_features_freeze.parquet` — 48-row team feature table
- `outputs/training_rows.parquet` — 896-row training dataset
- `outputs/prediction_rows.parquet` — 112-row prediction dataset
- `outputs/feature_audit_report.csv` — per-feature null counts, ranges, validation flags

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.leakage_guard import (
    check_freeze_date, check_elo_snapshot,
    check_no_synthetic_data, check_training_rows_chronological,
    check_tactical_gap_preserved, check_form_window,
    FREEZE_DATE, TACTICAL_GAP_TEAMS,
)
from src.name_map import CANONICAL_48, WC_DEBUTANTS, canonicalize
from src.features import (
    load_ds2, load_ds4, load_ds6, load_ds8, load_ds10,
    load_ds16, load_ds17, load_ds1,
    build_elo_features, build_fifa_features, build_wc_historical_features,
    build_form_features, build_tournament_features, build_penalty_features,
    build_team_features, build_feature_table, build_training_rows,
)

DATA_ROOT = PROJECT_ROOT
ARC_BASE  = DATA_ROOT / "archive.zip"
ARC2      = DATA_ROOT / "archive (2).zip"
ARC3      = DATA_ROOT / "archive (3).zip"
ARC4      = DATA_ROOT / "archive (4).zip"
ARC6      = DATA_ROOT / "archive (6).zip"
OUTPUTS   = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

print("Imports OK — loading raw datasets...")

## Cell 2 — Load data and run leakage guards

In [ ]:
ds2  = load_ds2(ARC4)
ds4  = load_ds4(ARC2)
ds6  = load_ds6(ARC6)
ds8  = load_ds8(ARC3)
ds10 = load_ds10(ARC2)
ds16 = load_ds16(ARC_BASE)
ds17 = load_ds17(ARC_BASE)
ds1  = load_ds1(ARC2)

# ── Leakage guards ────────────────────────────────────────────────────────
check_freeze_date(ds1, date_column="date")
check_freeze_date(ds4, date_column="date")
check_elo_snapshot(ds2)

form_dates = ds4[ds4["date"] >= "2024-01-01"]
check_form_window(form_dates)

print("✓ All leakage guards passed")
print(f"  DS2: {len(ds2)} Elo rows | DS4: {len(ds4)} match rows | DS8: {len(ds8)} WC rows")

## Cell 3 — Elo features

In [ ]:
elo_features = build_elo_features(ds2)

assert len(elo_features) == 48, f"Expected 48 Elo rows, got {len(elo_features)}"

# Design-spec spot checks
if "Spain" in elo_features.index:
    assert elo_features.loc["Spain", "elo_rating"] == 2165.0, "Spain Elo mismatch"
    print("✓ Spain elo_rating = 2165")

hosts = ["United States", "Canada", "Mexico"]
for h in hosts:
    if h in elo_features.index:
        assert elo_features.loc[h, "elo_is_host"] == 1, f"{h} should have is_host=1"
print(f"✓ Host flag verified for {[h for h in hosts if h in elo_features.index]}")

print("\nTop 10 teams by Elo rating:")
elo_features.sort_values("elo_rating", ascending=False).head(10)

## Cell 4 — FIFA features

In [ ]:
fifa_features = build_fifa_features(ds10, elo_features)

assert len(fifa_features) == 48, f"Expected 48 FIFA rows, got {len(fifa_features)}"

print("FIFA rank vs Elo rank disagreement (top 10):")
if "elo_fifa_rank_disagreement" in fifa_features.columns:
    fifa_features[["elo_fifa_rank_disagreement"]].abs().sort_values(
        "elo_fifa_rank_disagreement", ascending=False
    ).head(10)

## Cell 5 — WC historical features

In [ ]:
wc_features = build_wc_historical_features(ds8)

assert len(wc_features) == 48, f"Expected 48 WC-hist rows, got {len(wc_features)}"

if "wc_debut_modern_flag" in wc_features.columns:
    debut_count = int(wc_features["wc_debut_modern_flag"].sum())
    print(f"Teams with wc_debut_modern_flag = 1: {debut_count}")
    print("Expected debutants:", sorted(WC_DEBUTANTS))

    # Verify known debutants have the flag set
    for team in WC_DEBUTANTS:
        if team in wc_features.index:
            assert wc_features.loc[team, "wc_debut_modern_flag"] == 1, (
                f"{team} should have debut flag"
            )
    print(f"✓ All {len(WC_DEBUTANTS)} known debutants verified")

wc_features.head()

## Cell 6 — Form features

In [ ]:
form_features = build_form_features(ds4)

assert len(form_features) == 48, f"Expected 48 form rows, got {len(form_features)}"

print("Form features shape:", form_features.shape)

if "form_win_rate_last10" in form_features.columns:
    print("\nTop 10 by form win rate:")
    display(form_features[["form_win_rate_last10", "form_gd_last10"]]
            .sort_values("form_win_rate_last10", ascending=False).head(10))

## Cell 7 — Tournament features (MD1 + MD2 in-tournament signal)

In [ ]:
completed = ds1.dropna(subset=["home_score", "away_score"]).copy()

tourn_features = build_tournament_features(completed, ds17)

assert len(tourn_features) == 48, f"Expected 48 tournament rows, got {len(tourn_features)}"

# Design spec: exactly 4 teams have incomplete tactical data
if "has_full_tactical_md2" in tourn_features.columns:
    gap_teams = tourn_features[~tourn_features["has_full_tactical_md2"]].index.tolist()
    print(f"Teams with has_full_tactical_md2 = False: {sorted(gap_teams)}")
    for t in TACTICAL_GAP_TEAMS:
        if t in tourn_features.index:
            assert not tourn_features.loc[t, "has_full_tactical_md2"], (
                f"{t} should have incomplete tactical data"
            )
    print(f"✓ Tactical gap teams match design spec: {sorted(TACTICAL_GAP_TEAMS)}")

# Run leakage guard for tactical gap preservation
check_tactical_gap_preserved(tourn_features)
print("✓ check_tactical_gap_preserved passed")

if "tourn_pts_md2" in tourn_features.columns:
    print("\nTop teams by tournament points:")
    display(tourn_features[["tourn_pts_md2", "tourn_gf_md2", "tourn_gd_md2"]]
            .sort_values("tourn_pts_md2", ascending=False).head(8))

## Cell 8 — Penalty/shootout features

In [ ]:
penalty_features = build_penalty_features(ds6, ds8)

assert len(penalty_features) == 48, f"Expected 48 penalty rows, got {len(penalty_features)}"

# Design spec spot checks (k=8 Bayesian shrinkage)
if "shootout_win_rate_alltime" in penalty_features.columns:
    for team, expected in [("Germany", 0.625), ("England", 0.40)]:
        if team in penalty_features.index:
            actual = float(penalty_features.loc[team, "shootout_win_rate_alltime"])
            print(f"{team}: shrunk rate = {actual:.4f}  (expected ≈ {expected})")

if "shootout_naive_flag" in penalty_features.columns:
    naive_teams = penalty_features[penalty_features["shootout_naive_flag"] == 1].index.tolist()
    print(f"\nTeams using naive shootout prior (no WC history): {sorted(naive_teams)}")

## Cell 9 — Assemble full team feature table

In [ ]:
team_features = build_feature_table(
    ds1=completed, ds2=ds2, ds4=ds4, ds6=ds6, ds8=ds8, ds10=ds10,
    ds17=ds17,
    annex_c_path=DATA_ROOT / "third_place_annex_c.csv",
)

print(f"Team feature table shape: {team_features.shape}")
print(f"  (expected: 48 rows, ~55 columns)")

# Leakage guard on assembled table
check_no_synthetic_data(team_features)
print("✓ check_no_synthetic_data passed")

# Null counts
null_counts = team_features.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]
if not cols_with_nulls.empty:
    print(f"\nFeatures with nulls (≤12 expected for debutant WC features):")
    print(cols_with_nulls.to_string())
else:
    print("\n✓ No nulls in feature table")

team_features.head()

## Cell 10 — Build training rows

In [ ]:
training_rows = build_training_rows(ds8, ds4, ds2, ds10)

print(f"Training rows shape: {training_rows.shape}")
print(f"  (expected: 896 rows × 57 cols)")

assert len(training_rows) == 896, f"Expected 896 training rows, got {len(training_rows)}"

if "outcome" in training_rows.columns:
    dist = training_rows["outcome"].value_counts().sort_index()
    print(f"\nOutcome distribution: {dist.to_dict()}")
    print("  (expected ≈ WIN:272, DRAW:166, LOSS:234 — 2×448 WC matches)")

# L1 leakage guard: elo_year_used must equal match_year - 1
check_training_rows_chronological(training_rows)
print("\n✓ check_training_rows_chronological passed")

## Cell 11 — Feature audit report

In [ ]:
audit_rows = []
for col in team_features.columns:
    s = team_features[col]
    numeric_s = pd.to_numeric(s, errors="coerce")
    audit_rows.append({
        "feature":    col,
        "null_count": int(s.isnull().sum()),
        "dtype":      str(s.dtype),
        "min":        float(numeric_s.min()) if numeric_s.notna().any() else None,
        "max":        float(numeric_s.max()) if numeric_s.notna().any() else None,
        "mean":       float(numeric_s.mean()) if numeric_s.notna().any() else None,
    })

audit_df = pd.DataFrame(audit_rows)
print(f"Feature audit report: {len(audit_df)} features")

out_path = OUTPUTS / "feature_audit_report.csv"
audit_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

audit_df

## Cell 12 — Save outputs

In [ ]:
paths = {
    "team_features_freeze.parquet": team_features,
    "training_rows.parquet":        training_rows,
}

for filename, df in paths.items():
    path = OUTPUTS / filename
    df.to_parquet(path, index=True)
    print(f"Saved → {path}  shape={df.shape}")

print("\n✓ All feature engineering outputs saved")